In [67]:
# ── Generic Imports ────────────
import numpy as np 
from PIL import Image 
import scipy.io as sio
import matplotlib.pyplot as plt

import numpy as np
import networkx as nx
from scipy.spatial import distance_matrix
from scipy.optimize import linear_sum_assignment

In [68]:
# ── Constants ──────────────────
RESIZE: int = 256
DATA_PATH: str = "./../../data/WillowObject/WILLOW-ObjectClass/"

In [69]:
# ── Data Class ─────────────────
class NotatedImage:
    # 1. Constructor Method
    def __init__(self, img, kpts) -> None:
        self.img: Image.Image = img 
        self.kpts: np.array = kpts

    # 2. Resize Method
    def resize(self):
        self.kpts[0] *= RESIZE / self.img.size[0]
        self.kpts[1] *= RESIZE / self.img.size[1]
        self.img = self.img.resize((RESIZE, RESIZE), resample=Image.BILINEAR)
        return self

In [70]:
# Local Imports
import glob # For searching files
import os   # To remove file extension

# ── Path Retrieval Function ──
def getting(cat: str, n: int) -> list[NotatedImage]:
    images: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.png")
    points: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.mat")
    output = []
    set_points = set(points)

    for img_file in images:
        base, _ = os.path.splitext(img_file)
        mat_file = base + ".mat"
        if mat_file in set_points:
            img = Image.open(img_file)
            kpts = np.array(sio.loadmat(mat_file)['pts_coord'])
            ni = NotatedImage(img, kpts)
            output.append(ni)
            
            if n is not None and len(output) >= n:
                break

    return output

In [71]:
 # Local Import
from scipy.spatial import Delaunay

# ── Delaunay Function ─────────
def delaunay_graph(self):
    points = list(zip(self.kpts[0], self.kpts[1]))
    tri = Delaunay(points)
    edges = []

    for simplex in tri.simplices:
        for i in range(len(simplex)):
            for j in range(i + 1, len(simplex)):
                edges.append((simplex[i], simplex[j]))

    self.edges = list(set(tuple(sorted(e)) for e in edges))
    return self

# We append it to NotatedImage
NotatedImage.add_delaunay = delaunay_graph

In [72]:
# Local Import
import math

# ── KNN Function ─────────────
def knn_graph(self, k) -> list[tuple[int, int]]:
    # Initialize the Matrix where we store distances
    distance_matrix: list[list[float]] = []
    # We calculate the distance from each point to another
    points = list(zip(self.kpts[0], self.kpts[1]))
    for i, point in enumerate(points):
        # We initialize the array where we will store the distances for each point
        distance_array: list[float] = []
        # IMPORTANT: i and j are point indexes
        for j, other in enumerate(points):
            distance = math.sqrt((point[0]-other[0])**2 + (point[1]-other[1])**2) # Eucleadian distance
            distance_array.append((distance, j))
        distance_array.sort(key=lambda dist: dist[0])
        distance_matrix.append(distance_array.copy())
    
    # Initialize the Result Variable
    edges: list[tuple[int, int]] = []
    # We append the Edges (Each Row's Index ~ Point Index)
    for i, array in enumerate(distance_matrix):
        array = array[1:k+1] # Skip the distance to itself
        for element in array:
            edges.append((i, element[1])) # Build and append the array

    self.edges = list(edges)          
    return self

# We append it to NotatedImage
NotatedImage.add_knn = knn_graph


In [73]:
import numpy as np
import networkx as nx

def make_adj(self):
    n = self.kpts.shape[1]
    self.adj = np.zeros((n, n), dtype=int)
    for i, j in self.edges:
        self.adj[i, j] = 1
        self.adj[j, i] = 1 # Undirected
    return self

NotatedImage.make_adj = make_adj

In [74]:
class Pair:
    def __init__(self, ni_a, ni_b) -> None:
        self.ni_a: NotatedImage = ni_a 
        self.ni_b: NotatedImage = ni_b 

In [75]:
import numpy as np

def calculate_accuracy(self) -> float:
    n, m = self.match.shape
    ground_truth = np.eye(n, m) 
    
    # Count how many predicted matches fall exactly on the diagonal
    correct_matches = np.sum((self.match == 1) & (ground_truth == 1))
    
    self.acc = correct_matches / min(n, m)
    return self

# Append it to the class
Pair.get_accuracy = calculate_accuracy

In [76]:
import random
import networkx as nx
import numpy as np
from gensim.models import Word2Vec
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def compute_node2vec_embeddings(G, dimensions=64, num_walks=10, walk_length=30):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [node]
            while len(walk) < walk_length:
                neighbors = list(G.neighbors(walk[-1]))
                if not neighbors: break
                walk.append(random.choice(neighbors))
            walks.append([str(n) for n in walk])
            
    model = Word2Vec(sentences=walks, vector_size=dimensions, window=5, min_count=1, sg=1, workers=4)
    return np.array([model.wv[str(n)] for n in nodes])

In [77]:
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def enhanced_spatial_matching(self, k: int = 0, w_spatial: float = 0.6, w_n2v: float = 0.4):
    # 1. Build graphs AND their adjacency matrices safely
    if k > 0:
        self.ni_a.add_knn(k).make_adj()
        self.ni_b.add_knn(k).make_adj()
    else:
        self.ni_a.add_delaunay().make_adj()
        self.ni_b.add_delaunay().make_adj()
    
    # 2. Convert into NetworkX Graphs
    G1 = nx.from_numpy_array(self.ni_a.adj) 
    G2 = nx.from_numpy_array(self.ni_b.adj)

    # 3. Compute Node2Vec embeddings
    emb_a = compute_node2vec_embeddings(G1)
    emb_b = compute_node2vec_embeddings(G2)

    # 4. Format keypoints and calculate distance matrices
    points_a, points_b = self.ni_a.kpts.T, self.ni_b.kpts.T
    spatial_cost = cdist(points_a, points_b, metric='euclidean')
    topological_cost = cdist(emb_a, emb_b, metric='euclidean')

    # 5. Combine and solve
    cost_matrix = (w_spatial * spatial_cost) + (w_n2v * topological_cost)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # 6. Save matching matrix
    self.match = np.zeros_like(cost_matrix, dtype=int)
    self.match[row_ind, col_ind] = 1
    
    return self

# Bind it to the Pair object
Pair.add_match = enhanced_spatial_matching

In [78]:
# THE FUNCTION:
import itertools
import pandas as pd
import os

# 1. Update your visualization function to accept a save path
def visualize_matching_full(pa: Pair, save_path: str):
    img_a, img_b = pa.ni_a.img, pa.ni_b.img
    kpts_a, kpts_b = pa.ni_a.kpts, pa.ni_b.kpts
    
    width = max(img_a.size[0], img_b.size[0])
    height = max(img_a.size[1], img_b.size[1])
    composite = Image.new('RGB', (width * 2, height))
    composite.paste(img_a, (0, 0))
    composite.paste(img_b, (width, 0)) 
    
    plt.figure(figsize=(12, 6))
    plt.imshow(composite)
    plt.axis('off')
    
    for i, j in pa.ni_a.edges:
        plt.plot([kpts_a[0, i], kpts_a[0, j]], [kpts_a[1, i], kpts_a[1, j]], 'y-', alpha=0.5, lw=1)
    for i, j in pa.ni_b.edges:
        plt.plot([kpts_b[0, i] + width, kpts_b[0, j] + width], [kpts_b[1, i], kpts_b[1, j]], 'y-', alpha=0.5, lw=1)
        
    rows, cols = np.where(pa.match == 1)
    for i, j in zip(rows, cols):
        plt.plot([kpts_a[0, i], kpts_b[0, j] + width], [kpts_a[1, i], kpts_b[1, j]], 'g-', lw=1.5, alpha=0.8)
        
    plt.scatter(kpts_a[0], kpts_a[1], c='w', edgecolors='k', s=40, zorder=5)
    plt.scatter(kpts_b[0] + width, kpts_b[1], c='w', edgecolors='k', s=40, zorder=5)
    
    plt.title(f"Graph Matching Results - Acc: {pa.acc:.2f}")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

In [79]:
import pandas as pd
import itertools
import numpy as np

def evaluate_part_2():
    categories = ['Car', 'Duck', 'Face', 'Motorbike', 'Winebottle']
    results = []
    
    # FIX: 'k' is 0, not None
    configs = [
        {'type': 'Delaunay', 'k': 0, 'w_s': 1.0, 'w_n': 0.0}, 
        {'type': 'Delaunay', 'k': 0, 'w_s': 0.6, 'w_n': 0.4}, 
        {'type': 'Delaunay', 'k': 0, 'w_s': 0.4, 'w_n': 0.6}, 
    ]
    
    knn_cats = ['Duck']
    for k in [3, 5, 7]:
        configs.extend([
            {'type': f'KNN_{k}', 'k': k, 'w_s': 0.6, 'w_n': 0.4},
            {'type': f'KNN_{k}', 'k': k, 'w_s': 0.4, 'w_n': 0.6}
        ])

    for cat in categories:
        for config in configs:
            if 'KNN' in config['type'] and cat not in knn_cats:
                continue
                
            print(f"Running {cat} | {config['type']} | W_S: {config['w_s']}")
            
            # FIX: Load fresh images INSIDE the loop so they don't shrink infinitely
            imgs = getting(cat, None) 
            
            if len(imgs) < 2:
                continue
            
            for img in imgs:
                img.resize()
                # Use make_adj() here so the attribute exists
                if config['k'] > 0:
                    img.add_knn(config['k']).make_adj()
                else:
                    img.add_delaunay().make_adj()
                    
            pairs = [Pair(a, b) for a, b in itertools.combinations(imgs, 2)]
            
            best_pair, worst_pair = None, None
            best_acc, worst_acc = -1, 2
            
            for p in pairs:
                p.add_match(k=config['k'], w_spatial=config['w_s'], w_n2v=config['w_n'])
                p.get_accuracy()
                
                if p.acc > best_acc: best_acc, best_pair = p.acc, p
                if p.acc < worst_acc: worst_acc, worst_pair = p.acc, p
            
            if best_pair is None:
                continue

            accs = [p.acc for p in pairs]
            results.append({
                'Categoria': cat,
                'Tipo_Grafo': config['type'],
                'Peso_Espacial': config['w_s'],
                'Peso_Node2Vec': config['w_n'],
                'Acc': np.mean(accs),
                'Std': np.std(accs),
                'Num_Pares': len(pairs)
            })
            
            filename_base = f"{cat}_{config['type']}_wS{config['w_s']}"
            visualize_matching_full(best_pair, f"{filename_base}_good.png")
            visualize_matching_full(worst_pair, f"{filename_base}_bad.png")

    pd.DataFrame(results).to_csv('results.csv', index=False)
    print("Done! CSV and all combinations generated.")


In [80]:
evaluate_part_2()

Running Car | Delaunay | W_S: 1.0
Running Car | Delaunay | W_S: 0.6
Running Car | Delaunay | W_S: 0.4
Running Duck | Delaunay | W_S: 1.0
Running Duck | Delaunay | W_S: 0.6
Running Duck | Delaunay | W_S: 0.4
Running Duck | KNN_3 | W_S: 0.6
Running Duck | KNN_3 | W_S: 0.4
Running Duck | KNN_5 | W_S: 0.6
Running Duck | KNN_5 | W_S: 0.4
Running Duck | KNN_7 | W_S: 0.6
Running Duck | KNN_7 | W_S: 0.4
Running Face | Delaunay | W_S: 1.0
Running Face | Delaunay | W_S: 0.6
Running Face | Delaunay | W_S: 0.4
Running Motorbike | Delaunay | W_S: 1.0
Running Motorbike | Delaunay | W_S: 0.6
Running Motorbike | Delaunay | W_S: 0.4
Running Winebottle | Delaunay | W_S: 1.0
Running Winebottle | Delaunay | W_S: 0.6
Running Winebottle | Delaunay | W_S: 0.4
Done! CSV and all combinations generated.


Baseline:
Los resultados base usando solo información espacial (Delaunay con peso 1.0) muestran un rendimiento muy sólido, variando desde un 72.2% de precisión en 'Car' hasta un excelente 92.3% en 'Winebottle'.

Delaunay Optimizado:
Al integrar las características estructurales de Node2Vec (pesos 0.6/0.4 y 0.4/0.6) en el Delaunay optimizado, el rendimiento se mantiene prácticamente estático respecto a la baseline. Se observan mejoras microscópicas de apenas décimas de porcentaje en casi todas las categorías, lo que sugiere que la distancia espacial sigue dominando por completo el coste del emparejamiento.

KNN:
En la categoría 'Duck', el uso de topologías KNN (k=3, 5 y 7) tampoco presenta cambios significativos frente al Delaunay optimizado, manteniéndose ambos en un ~77.2%. Aunque la configuración KNN_5 (0.4/0.6) alcanza el pico máximo absoluto de precisión (77.31%), la diferencia es estadísticamente insignificante, demostrando que el método es poco sensible al tipo de grafo seleccionado.